In [12]:
import numpy as np
import pandas as pd

In [13]:
temp_df=pd.read_csv('IMDB Dataset.csv')
df=temp_df.iloc[:10000]

In [14]:
df.head(5)

,review,sentiment
0,One of the other reviewers has mentioned that ...,positive
1,A wonderful little production. <br /><br />The...,positive
2,I thought this was a wonderful way to spend ti...,positive
3,Basically there's a family where a little boy ...,negative
4,"Petter Mattei's ""Love in the Time of Money"" is...",positive


In [15]:
df.drop_duplicates(inplace=True)

In [16]:
import re
def remove_tags(raw_text):
    cleaned_text=re.sub(re.compile('<.*?>'), '', raw_text)
    return cleaned_text

In [17]:
df['review']=df['review'].apply(remove_tags)

In [18]:
df

,review,sentiment
0,One of the other reviewers has mentioned that ...,positive
1,A wonderful little production. The filming tec...,positive
2,I thought this was a wonderful way to spend ti...,positive
3,Basically there's a family where a little boy ...,negative
4,"Petter Mattei's ""Love in the Time of Money"" is...",positive
...,...,...
9995,"Fun, entertaining movie about WWII German spy ...",positive
9996,Give me a break. How can anyone say that this ...,negative
9997,This movie is a bad movie. But after watching ...,negative
9998,This is a movie that was probably made to ente...,negative


In [19]:
df['review']=df['review'].apply(lambda x:x.lower())
df

,review,sentiment
0,one of the other reviewers has mentioned that ...,positive
1,a wonderful little production. the filming tec...,positive
2,i thought this was a wonderful way to spend ti...,positive
3,basically there's a family where a little boy ...,negative
4,"petter mattei's ""love in the time of money"" is...",positive
...,...,...
9995,"fun, entertaining movie about wwii german spy ...",positive
9996,give me a break. how can anyone say that this ...,negative
9997,this movie is a bad movie. but after watching ...,negative
9998,this is a movie that was probably made to ente...,negative


In [20]:
from nltk.corpus import stopwords
sw_list=stopwords.words('english')

In [21]:
def remove_stopwords(text):
    new_text = []
    
    for word in text.split():
        if word in sw_list:
            new_text.append('')
        else:
            new_text.append(word)
    x=new_text[:]
    new_text.clear()
    return " ".join(x) # Join the list of words into a single string and return it

In [22]:
df['review'] = df['review'].apply(remove_stopwords)

In [23]:
import gensim

In [24]:
from nltk.tokenize import sent_tokenize
from gensim.utils import simple_preprocess

In [26]:
story = []

for doc in df['review']:
    sentences = sent_tokenize(doc)
    for sentence in sentences:
        story.append(simple_preprocess(sentence))

In [27]:
model=gensim.models.Word2Vec(window=10, min_count=2)

In [28]:
model.build_vocab(story)

In [29]:
model.train(story, total_examples=model.corpus_count, epochs=model.epochs)

(5850056, 6186875)

In [30]:
len(model.wv.index_to_key)

31845

In [32]:
def document_vector(doc):
    doc = [word for word in doc.split() if word in model.wv.index_to_key]
    return np.mean(model.wv[doc], axis=0)

In [33]:
document_vector(df['review'].values[0])

array([-0.00853615,  0.2935393 ,  0.16011293, -0.11857355, -0.07269597,
       -0.47829127,  0.15178019,  0.7896925 , -0.19041213, -0.1259535 ,
       -0.09712408, -0.54809934,  0.01260428,  0.21384594,  0.01999118,
       -0.25247413,  0.18082331, -0.5606406 ,  0.01627094, -0.698268  ,
        0.11763768,  0.12331481,  0.28244072, -0.0999032 , -0.24095364,
       -0.12322108, -0.22774367, -0.29014933, -0.28034124, -0.04731968,
        0.41566953, -0.02215296,  0.01730236, -0.3552936 , -0.2260732 ,
        0.27909884,  0.14408414, -0.41056275, -0.25386938, -0.72104746,
        0.18643126, -0.33389914, -0.333093  , -0.13776284,  0.40574083,
       -0.1721165 , -0.39440402, -0.10835983,  0.17827357,  0.20978834,
        0.09987029, -0.28172836, -0.247999  , -0.1750392 , -0.3232323 ,
        0.12260741,  0.42638063,  0.09917051, -0.2997477 ,  0.07615168,
       -0.00711567,  0.11415814, -0.12673253,  0.04754496, -0.4262456 ,
        0.39349163, -0.02429097,  0.13974844, -0.65945786,  0.32

In [34]:
from tqdm import tqdm

In [35]:
x=[]
for doc in tqdm(df['review'].values):
    x.append(document_vector(doc))

100%|██████████| 9983/9983 [03:40<00:00, 45.30it/s]


In [36]:
x=np.array(x)

In [37]:
x.shape

(9983, 100)

In [38]:
from sklearn.preprocessing import LabelEncoder
le=LabelEncoder()
y=le.fit_transform(df['sentiment'])

In [39]:
y

array([1, 1, 1, ..., 0, 0, 1], shape=(9983,))

In [40]:
from sklearn.model_selection import train_test_split
x_train,x_test,y_train,y_test=train_test_split(x,y,test_size=0.2,random_state=1)

In [46]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score,classification_report,confusion_matrix


In [47]:
model=RandomForestClassifier()
model.fit(x_train,y_train)
y_pred=model.predict(x_test)
accuracy_score(y_test,y_pred)


0.7771657486229344